In [11]:
import json
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

#### Data ingest and transform

In [12]:
# DATA INGEST: 

# Load the provided JSON file
file_path = 'DE_L_results_T_10_delta_5_scen_4_trial_1_inv_1_cap._1_cap.inc._2 1.json'
with open(file_path, 'r') as file:
    data = json.load(file)
    
# Create the combined data dictionary according to the provided mapping
combined_data = {
    "F_tender_schedule": data.get("F", {}),
    "Y_manufacturers_producing_period": data.get("Y", {}),
    "W_manufacturers_participate_tender": data.get("W", {}),
    "L_capacity_extension_decision": data.get("L", {}),
    "Q_commitment_amounts": data.get("Q", {}),
    "X_production_amounts": data.get("X", {}),
    "I_inventory_level": data.get("I", {}),
    "Vc_vaccinated_children": data.get("Vc", {}),
    "S_unvaccinated_children": data.get("S", {})
}

# unique_producers = set(combined_data['Y_manufacturers_producing_period'].keys())
# unique_vaccines = set(combined_data['Q_commitment_amounts'].keys())

# # Load the scenario pair probabilities data from the provided file
# scenario_probabilities_file_path = 'scenario_pair_probabilities_140.json'
# with open(scenario_probabilities_file_path, 'r') as file:
#     scenario_probabilities = json.load(file)
    
# Load the uploaded Excel file
file_path = '../../data/production_capacity_scenarios/production_capacity_scenarios.xlsx'
xls = pd.ExcelFile(file_path)

# Read the "master capacity" sheet
df_master_capacity = pd.read_excel(xls, sheet_name='master_capacity')
# df_master_capacity = df_master_capacity.sort_values(by='Manufacturer')
df_master_capacity.set_index(df_master_capacity.columns[0], inplace=False)

df_master_capacity

,Manufacturer,1,2,3,4,5,6,7,8,9,10
0,AJ_Vaccines,7711003,7711003,7711003,7711003,7711003,7711003,7711003,7711003,7711003,7711003
1,BB_NCIPD,39223956,39223956,39223956,39223956,39223956,39223956,39223956,39223956,39223956,39223956
2,Bharat_Biotech,61029105,61029105,61029105,61029105,61029105,61029105,61029105,61029105,61029105,61029105
3,Bilthoven,12048153,12048153,12048153,12048153,12048153,12048153,12048153,12048153,12048153,12048153
4,Biological_E,164885690,164885690,164885690,164885690,164885690,164885690,164885690,164885690,164885690,164885690
5,China_National,12812242,12812242,12812242,12812242,12812242,12812242,12812242,12812242,12812242,12812242
6,GSK,226762686,226762686,226762686,226762686,226762686,226762686,226762686,226762686,226762686,226762686
7,Haffkine_Bio,80972855,80972855,80972855,80972855,80972855,80972855,80972855,80972855,80972855,80972855
8,LG_Chem,43702188,43702188,43702188,43702188,43702188,43702188,43702188,43702188,43702188,43702188
9,Merck_Sharp,56991052,56991052,56991052,56991052,56991052,56991052,56991052,56991052,56991052,56991052


In [22]:
combined_data['F_tender_schedule']

# dict(sorted(combined_data['L_capacity_extension_decision']['BB_NCIPD'].items()))

# {v for data in combined_data['L_capacity_extension_decision'].values() for v in data.values()}

{'PCV': {'5': {'5': 0.0, '6': 0.0, '7': 0.0, '9': -0.0, '8': 0.0},
  '4': {'5': 0.0, '4': 0.0, '6': 0.0, '7': 0.0, '8': 1.0},
  '6': {'6': 0.0, '7': 0.0, '10': 0.0, '9': 0.0, '8': 0.0},
  '7': {'7': 0.0, '10': 0.0, '9': 0.0, '8': 0.0},
  '2': {'5': 0.0, '4': 0.0, '6': 0.0, '2': 0.0, '3': -0.0},
  '10': {'10': 0.0},
  '9': {'10': 1.0, '9': 0.0},
  '8': {'10': 0.0, '9': 0.0, '8': 0.0},
  '3': {'5': 0.0, '4': 0.0, '6': 0.0, '7': 0.0, '3': 0.0},
  '1': {'5': 0.0, '4': 0.0, '2': 0.0, '3': 1.0, '1': 0.0}},
 'Measles': {'5': {'5': 0.0, '6': 0.0, '7': 0.0, '9': 0.0, '8': 0.0},
  '4': {'5': 0.0, '4': 0.0, '6': 0.0, '7': 0.0, '8': 1.0},
  '6': {'6': 0.0, '7': 0.0, '10': 0.0, '9': 0.0, '8': 0.0},
  '7': {'7': 0.0, '10': 0.0, '9': 0.0, '8': 0.0},
  '2': {'5': 0.0, '4': 0.0, '6': 0.0, '2': 0.0, '3': 0.0},
  '10': {'10': 0.0},
  '9': {'10': 1.0, '9': 0.0},
  '8': {'10': 0.0, '9': 0.0, '8': 0.0},
  '3': {'5': 0.0, '4': 0.0, '6': 0.0, '7': 0.0, '3': 0.0},
  '1': {'5': 0.0, '4': 0.0, '2': 0.0, '3': 1.0

#### L capacity increase

In [14]:
# Translate L capacity increase to useable data

# Adjusting the ordering function to handle nested dictionaries with non-numeric keys
def order_data(data_dict):
    ordered_data = {}
    for key, value in data_dict.items():
        if isinstance(value, dict):
            ordered_data[key] = {k: order_data(v) if isinstance(v, dict) else v for k, v in sorted(value.items(), key=lambda item: int(item[0]) if item[0].isdigit() else item[0])}
        else:
            ordered_data[key] = value
    return ordered_data

# Apply ordering to combined_data
L_ordered = order_data(combined_data['L_capacity_extension_decision'])

# Convert the 'L_capacity_extension_decision' data into a DataFrame for better visualization
L_df = pd.DataFrame(L_ordered).sort_index()
# Rotate the DataFrame
L_df_rotated = L_df.transpose()
# Order the columns from 1 to 10
ordered_columns = [str(i) for i in range(1, 11)]
L_df_rotated_ordered = L_df_rotated[ordered_columns]
# Reset the index to make manufacturers the first column
L_df_rotated_ordered.reset_index(inplace=True)
L_df_rotated_ordered.rename(columns={'index': 'Manufacturer'}, inplace=True)

L_df_rotated_ordered = L_df_rotated_ordered.sort_values(by='Manufacturer').reset_index(drop=True)
L_df_rotated_ordered
def transform_row(row, factor):
    row = row.copy()
    if row.iloc[1] == factor:
        row.iloc[1] = (1+ factor/10)
    else:
        row.iloc[1] = 1.0

    for i in range(2, len(row)):
        if row.iloc[i] == factor:
            row.iloc[i] = row.iloc[i-1] + (factor/10)
        else:
            row.iloc[i] = row.iloc[i-1]
    
    return row

def transform_row(row):
    row = row.copy()
    if row.iloc[1] == 2:
        row.iloc[1] = 1 + 0.2
    else:
        row.iloc[1] = 1.0

    for i in range(2, len(row)):
        if row.iloc[i] == 2:
            row.iloc[i] = row.iloc[i-1] + 0.2
        else:
            row.iloc[i] = row.iloc[i-1]
    
    return row

# Apply the transformation to each row
transformed_capacity_increase = L_df_rotated_ordered.apply(transform_row, axis=1)
# Rename all columns except the first one to integers
transformed_capacity_increase.rename(columns={col: int(col) for col in transformed_capacity_increase.columns[1:]}, inplace=True)
transformed_capacity_increase.set_index(transformed_capacity_increase.columns[0], inplace=False)

# Performing the multiplication
adjusted_capacity = df_master_capacity.iloc[:, 1:11] * (transformed_capacity_increase.iloc[:, 1:11]/1000)

# Adding the Manufacturer column back to the selected dataframe
adjusted_capacity['Manufacturer'] = df_master_capacity['Manufacturer']

adjusted_capacity = adjusted_capacity[[adjusted_capacity.columns[-1]] + list(adjusted_capacity.columns[:-1])]

adjusted_capacity.set_index(adjusted_capacity.columns[0], inplace=False)
adjusted_capacity

C:\Users\nicho\AppData\Local\Temp\ipykernel_27116\2263093232.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  L_df_rotated_ordered.rename(columns={'index': 'Manufacturer'}, inplace=True)


,Manufacturer,1,2,3,4,5,6,7,8,9,10
0,AJ_Vaccines,7711.0030,7711.0030,7711.0030,7.711003e+03,7.711003e+03,7.711003e+03,7.711003e+03,7.711003e+03,7.711003e+03,7.711003e+03
1,BB_NCIPD,47068.7472,54913.5384,62758.3296,7.060312e+04,7.844791e+04,7.844791e+04,7.844791e+04,7.844791e+04,7.844791e+04,7.844791e+04
2,Bharat_Biotech,73234.9260,85440.7470,97646.5680,1.098524e+05,1.220582e+05,1.220582e+05,1.220582e+05,1.220582e+05,1.220582e+05,1.220582e+05
3,Bilthoven,12048.1530,12048.1530,12048.1530,1.204815e+04,1.204815e+04,1.204815e+04,1.204815e+04,1.204815e+04,1.204815e+04,1.204815e+04
4,Biological_E,197862.8280,230839.9660,263817.1040,2.967942e+05,3.297714e+05,3.627485e+05,3.957257e+05,4.287028e+05,4.287028e+05,4.287028e+05
5,China_National,15374.6904,15374.6904,15374.6904,1.537469e+04,1.537469e+04,1.537469e+04,1.537469e+04,1.537469e+04,1.537469e+04,1.537469e+04
6,GSK,272115.2232,317467.7604,362820.2976,4.081728e+05,4.535254e+05,4.988779e+05,5.442304e+05,5.895830e+05,6.349355e+05,6.349355e+05
7,Haffkine_Bio,80972.8550,80972.8550,80972.8550,8.097285e+04,8.097285e+04,8.097285e+04,8.097285e+04,8.097285e+04,8.097285e+04,8.097285e+04
8,LG_Chem,52442.6256,61183.0632,69923.5008,7.866394e+04,8.740438e+04,8.740438e+04,8.740438e+04,8.740438e+04,8.740438e+04,8.740438e+04
9,Merck_Sharp,56991.0520,56991.0520,56991.0520,5.699105e+04,5.699105e+04,5.699105e+04,5.699105e+04,5.699105e+04,5.699105e+04,5.699105e+04


In [15]:
adjusted_capacity.to_excel('adjusted_capacity.xlsx', index=False)